## Ep 03 -- Structured Output with Pydantic

In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv
from pydantic import BaseModel,Field
from langchain.chat_models import init_chat_model



In [2]:
load_dotenv()

True

In [3]:
OLLAMA_MODEL = os.getenv("OLLAMA_MODEL","ollama:llama3.2")
GROQ_MODEL = os.getenv("GROPQ_MODEL","GROPQ:meta-llama/llama-prompt-guard-2-22m")

print("Ollama Model: ",OLLAMA_MODEL)
print("Groq Model:",GROQ_MODEL)

Ollama Model:  ollama:llama3.2
Groq Model: groq:qwen/qwen3.8-27b


##  Output Structure format using Pydantic

In [4]:
class Resume(BaseModel):
    name :str = Field(description = "Full name of the candidate")
    email : str | None = Field(description = "Get email if present,else None")
    years_experience: float = Field(description = "Total year of professional experience")
    skills : list[str] = Field(description ="List of techinical skills mentions")

In [5]:
Resume(name="arti",email="arti@123.com",years_experience=6,skills=["python","langchain","langgraph"])

Resume(name='arti', email='arti@123.com', years_experience=6.0, skills=['python', 'langchain', 'langgraph'])

In [6]:
# Resume(name='arti', email='arti@123.com', years_experience=6.0, skills=['python', 'langchain', 'langgraph'])


### Load Input Data

In [7]:
SAMPLE_DIR = Path("samples")

text = (SAMPLE_DIR/"resume_messy.txt").read_text(encoding="utf-8")

In [8]:
text

'hey so my name is Arti Borkar, been working as a backend dev for about 4 and a half\nyears now. you can reach me at arti.borkar@example.com or just ping on linkedin.\nmostly i do python, fastapi, postgres, some docker and aws. recently started learning\nlanggraph and building ai agents which is super fun. also know a bit of react but not my\nmain thing. looking for agentic ai engineer roles now.'

In [9]:
print(text)

hey so my name is Arti Borkar, been working as a backend dev for about 4 and a half
years now. you can reach me at arti.borkar@example.com or just ping on linkedin.
mostly i do python, fastapi, postgres, some docker and aws. recently started learning
langgraph and building ai agents which is super fun. also know a bit of react but not my
main thing. looking for agentic ai engineer roles now.


In [10]:
SAMPLE_DIR

WindowsPath('samples')

## Parse Resume Using LLM 

In [13]:
llm = init_chat_model(OLLAMA_MODEL , temperature=0)  #temperature = 0 matlab hamra text file same rahegi jb hum 1 lenge to us txt file mai changes or advance ban jayega

In [15]:
prompt = f"From given resume {text} \n extract name,email,years of experience and skills in json format."
responce = llm.invoke(prompt)

In [16]:
responce

AIMessage(content='Here is the extracted information in JSON format:\n\n```\n{\n  "name": "Arti Borkar",\n  "email": "arti.borkar@example.com",\n  "yearsOfExperience": 4.5,\n  "skills": [\n    "Python",\n    "FastAPI",\n    "Postgres",\n    "Docker",\n    "AWS",\n    "LangGraph",\n    "React"\n  ]\n}\n```\n\nLet me know if you\'d like me to add anything else!', additional_kwargs={}, response_metadata={'model': 'llama3.2', 'created_at': '2026-09-26T13:44:54.1384381Z', 'done': True, 'done_reason': 'stop', 'total_duration': 16415862200, 'load_duration': 292495600, 'prompt_eval_count': 141, 'prompt_eval_duration': 756221000, 'eval_count': 102, 'eval_duration': 15267619000, 'logprobs': None, 'model_name': 'llama3.2', 'model_provider': 'ollama'}, id='lc_run--01a0ddf5-f03e-70d0-956c-dbc4964ff0ea-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 141, 'output_tokens': 102, 'total_tokens': 243})

In [17]:
responce.content

'Here is the extracted information in JSON format:\n\n```\n{\n  "name": "Arti Borkar",\n  "email": "arti.borkar@example.com",\n  "yearsOfExperience": 4.5,\n  "skills": [\n    "Python",\n    "FastAPI",\n    "Postgres",\n    "Docker",\n    "AWS",\n    "LangGraph",\n    "React"\n  ]\n}\n```\n\nLet me know if you\'d like me to add anything else!'

In [18]:
llm_structure = llm.with_structured_output(Resume , include_raw = True)

In [24]:
llm_structure.invoke(prompt)

{'raw': AIMessage(content='{"name":"Arti Borkar","email":"arti.borkar@example.com","years_experience":4.5,"skills":["python","fastapi","postgres","docker","aws","langgraph","react"]}', additional_kwargs={}, response_metadata={'model': 'llama3.2', 'created_at': '2026-09-26T14:10:57.6820969Z', 'done': True, 'done_reason': 'stop', 'total_duration': 6373303400, 'load_duration': 18683000, 'prompt_eval_count': 141, 'prompt_eval_duration': 237471000, 'eval_count': 44, 'eval_duration': 6103523000, 'logprobs': None, 'model_name': 'llama3.2', 'model_provider': 'ollama'}, id='lc_run--01a0de0d-f30d-7981-bae2-87d1fb44efb4-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 141, 'output_tokens': 44, 'total_tokens': 185}),
 'parsed': Resume(name='Arti Borkar', email='arti.borkar@example.com', years_experience=4.5, skills=['python', 'fastapi', 'postgres', 'docker', 'aws', 'langgraph', 'react']),
 'parsing_error': None}

## Messages : System vs User

system messge = the agent's role/rules("you are a precise data extractor.).
User messgse = the actual request/content .Good system prompt = more reliable output.(we'll harden these as guardrails in ep 35.)

In [25]:
system = "You are a precise data extractor. Extract only what is present;never invent data."

messages = [
    {"role":"system","content":system},
    {"role":"user","content": f"Extract structure data from this resume :\n {text}"}

]

responce = llm_structure.invoke(messages)

In [26]:
responce

{'raw': AIMessage(content='{\n  "name": "Arti Borkar",\n  "email": "arti.borkar@example.com",\n  "years_experience": 4.5,\n  "skills": [\n    "Python",\n    "FastAPI",\n    "Postgres",\n    "Docker",\n    "AWS",\n    "LangGraph",\n    "React"\n  ]\n}', additional_kwargs={}, response_metadata={'model': 'llama3.2', 'created_at': '2026-09-26T14:11:25.2936909Z', 'done': True, 'done_reason': 'stop', 'total_duration': 15120771500, 'load_duration': 70511900, 'prompt_eval_count': 148, 'prompt_eval_duration': 3986921000, 'eval_count': 75, 'eval_duration': 11004905000, 'logprobs': None, 'model_name': 'llama3.2', 'model_provider': 'ollama'}, id='lc_run--01a0de0e-3ccf-7603-a617-d4663df2d063-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 148, 'output_tokens': 75, 'total_tokens': 223}),
 'parsed': Resume(name='Arti Borkar', email='arti.borkar@example.com', years_experience=4.5, skills=['Python', 'FastAPI', 'Postgres', 'Docker', 'AWS', 'LangGraph', 'React']),
 'parsing_erro

In [27]:
type(responce)

dict

In [28]:
responce["parsed"]

Resume(name='Arti Borkar', email='arti.borkar@example.com', years_experience=4.5, skills=['Python', 'FastAPI', 'Postgres', 'Docker', 'AWS', 'LangGraph', 'React'])

## TOKEN/COUNT AWARENESS

{'raw': AIMessage(content='{\n  "name": "Arti Borkar",\n  "email": "arti.borkar@example.com",\n  "years_experience": 4.5,\n  "skills": [\n    "Python",\n    "FastAPI",\n    "Postgres",\n    "Docker",\n    "AWS",\n    "LangGraph",\n    "React"\n  ]\n}', additional_kwargs={}, response_metadata={'model': 'llama3.2', 'created_at': '2026-09-26T14:11:25.2936909Z', 'done': True, 'done_reason': 'stop', 'total_duration': 15120771500, 'load_duration': 70511900, 'prompt_eval_count': 148, 'prompt_eval_duration': 3986921000, 'eval_count': 75, 'eval_duration': 11004905000, 'logprobs': None, 'model_name': 'llama3.2', 'model_provider': 'ollama'}, id='lc_run--01a0de0e-3ccf-7603-a617-d4663df2d063-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 148, 'output_tokens': 75, 'total_tokens': 223}),

In [29]:
usage = getattr(responce.get("raw"),"usage_metadata",None)
if usage:
    print(f"tokens - in : {usage.get('input_tokens')} out: {usage.get('output_tokens')}"
            f"total: {usage.get('total_tokens')}")

tokens - in : 148 out: 75total: 223


## Never Trust Unvalidation output

In [42]:
if responce.get("parsing_error"):
    print("Model output failed validation; handle/retry instead of trusting it.")
else:
    resume = responce.get("parsed")
    print(resume.model_dump_json(indent=2))
    print(f"\nValidation! {len(resume.skills)} skills, {resume.years_experience} yrs experience.")

{
  "name": "Arti Borkar",
  "email": "arti.borkar@example.com",
  "years_experience": 4.5,
  "skills": [
    "Python",
    "FastAPI",
    "Postgres",
    "Docker",
    "AWS",
    "LangGraph",
    "React"
  ]
}

Validation! 7 skills, 4.5 yrs experience.
